In [ ]:
!pip install pyspark pandas

: 

# Import Library dan Setup Spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, max as spark_max, min as spark_min, avg, window, to_timestamp, lower, when, lag, date_format, lit
from pyspark.sql.window import Window

# Inisialisasi Spark Session
spark = SparkSession.builder \
    .appName("Analisis_Harga_Pangan_Kelompok4") \
    .getOrCreate()

print("Spark Session Berhasil Dibuat!")

# Load Data dari HDFS

In [ ]:
# Ganti localhost menjadi host.docker.internal
HDFS_API_PATH = "hdfs://host.docker.internal:9000/data/pangan/api/*"
HDFS_RSS_PATH = "hdfs://host.docker.internal:9000/data/pangan/rss/*"

try:
    # Load Data Harga Pangan
    df_api = spark.read.json(HDFS_API_PATH)
    df_api = df_api.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd'T'HH:mm:ss"))
    
    print("Skema Data Harga (API):")
    df_api.printSchema()
except Exception as e:
    print("Data API belum tersedia di HDFS:", e)

try:
    # Load Data Berita RSS
    df_rss = spark.read.json(HDFS_RSS_PATH)
    if "timestamp" in df_rss.columns:
        df_rss = df_rss.withColumn("timestamp", to_timestamp(col("timestamp")))
    elif "published" in df_rss.columns:
        df_rss = df_rss.withColumn("timestamp", to_timestamp(col("published")))
        
    print("Skema Data RSS:")
    df_rss.printSchema()
except Exception as e:
    print("Data RSS belum tersedia di HDFS:", e)

# Analisisis  Votalitas Harga

In [ ]:
# Menghitung volatilitas: (max - min) / avg * 100
df_volatilitas = df_api.groupBy("komoditas").agg(
    spark_max("harga").alias("harga_max"),
    spark_min("harga").alias("harga_min"),
    avg("harga").alias("harga_avg")
)

df_volatilitas = df_volatilitas.withColumn(
    "volatilitas_persen",
    ((col("harga_max") - col("harga_min")) / col("harga_avg")) * 100
).orderBy(col("volatilitas_persen").desc())

print("=== Ranking Komoditas Paling Fluktuatif ===")
df_volatilitas.show()

# Analisis Tren Harga

In [ ]:
# Grouping berdasarkan Komoditas dan Waktu (Per jam)
df_api_waktu = df_api.withColumn("waktu_jam", date_format(col("timestamp"), "yyyy-MM-dd HH:00:00"))

df_tren = df_api_waktu.groupBy("komoditas", "waktu_jam").agg(
    avg("harga").alias("harga_rata_rata")
)

# Window Function untuk melihat pergerakan naik/turun
windowSpec = Window.partitionBy("komoditas").orderBy("waktu_jam")
df_tren = df_tren.withColumn("harga_sebelumnya", lag("harga_rata_rata", 1).over(windowSpec))

# Logic Naik/Turun
df_tren = df_tren.withColumn(
    "status_tren",
    when(col("harga_sebelumnya").isNull(), "Awal")
    .when(col("harga_rata_rata") > col("harga_sebelumnya"), "Naik")
    .when(col("harga_rata_rata") < col("harga_sebelumnya"), "Turun")
    .otherwise("Stabil")
)

print("=== Tren Harga Komoditas ===")
df_tren.orderBy("waktu_jam", "komoditas").show()

# Analisis Koerelasi Berita

In [ ]:
# Daftar kata kunci komoditas yang akan dicari di judul berita
list_komoditas = ["beras", "cabai", "bawang", "daging", "telur", "ayam"]

# Samakan format waktu di data RSS
df_rss_waktu = df_rss.withColumn("waktu_jam", date_format(col("timestamp"), "yyyy-MM-dd HH:00:00"))
df_rss_lower = df_rss_waktu.withColumn("judul_lower", lower(col("title"))) # Asumsi kolomnya 'title'

korelasi_df = None
for k in list_komoditas:
    # Filter berita yang mengandung kata kunci komoditas di judulnya
    df_k = df_rss_lower.filter(col("judul_lower").contains(k)) \
                       .groupBy("waktu_jam") \
                       .count() \
                       .withColumnRenamed("count", "jumlah_berita") \
                       .withColumn("komoditas", lit(k))
    
    if korelasi_df is None:
        korelasi_df = df_k
    else:
        korelasi_df = korelasi_df.union(df_k)

# Gabungkan Data Tren Harga dan Data Berita berdasarkan Komoditas & Waktu
if korelasi_df is not None:
    df_korelasi_akhir = df_tren.join(korelasi_df, on=["komoditas", "waktu_jam"], how="left").fillna(0)
    print("=== Korelasi Tren Harga & Eksposur Berita ===")
    df_korelasi_akhir.orderBy("waktu_jam", "komoditas").show()
else:
    # Fallback jika data RSS kosong
    df_korelasi_akhir = df_tren.withColumn("jumlah_berita", lit(0))

# Output Data dan Export untuk Dashboard 

In [ ]:
import os
import json

print("Memulai proses export data ke JSON...")

# Membuat folder dashboard/data jika belum ada
PATH_DASHBOARD = "../dashboard/data"
os.makedirs(PATH_DASHBOARD, exist_ok=True)

# Konversi DataFrame Spark ke Pandas, lalu ke Dictionary (Format JSON)
hasil_export = {
    "volatilitas": df_volatilitas.toPandas().to_dict(orient="records"),
    "tren_harga": df_tren.toPandas().to_dict(orient="records"),
    "korelasi_berita": df_korelasi_akhir.toPandas().to_dict(orient="records")
}

# Simpan sebagai file JSON
file_path = f"{PATH_DASHBOARD}/spark_results.json"
with open(file_path, "w") as f:
    json.dump(hasil_export, f, indent=4)

print(f"File berhasil diexport untuk folder:\n{os.path.abspath(file_path)}")